# ModernBERT + LoRA numeracy: train → evaluate → registry, one session
All logic lives in the `training/` package in this repo; this notebook is only the entry
point. Sections:
1. **Setup** — mount Drive, sync repo, install deps
2. **Config** — one `RunConfig` cell, the only thing to edit between experiments
3. **Train** — `train_one_run()`
4. **Evaluate** — triplet metrics + ordering suites on the fresh adapter (or any past run)
5. **Record & compare** — append to `runs/registry.jsonl`, view all runs side by side

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/numeric_finetune_data'
os.makedirs(WORK_DIR, exist_ok=True)


In [ ]:
# One-time clone, then git pull on subsequent runs so this notebook always
# uses the latest training/ code without re-copying it into every notebook.
REPO_DIR = os.path.join(WORK_DIR, 'repo')
REPO_URL = 'git@github.com:<you>/Embed_improvements.git'  # TODO: set your repo URL

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

import sys
sys.path.insert(0, REPO_DIR)


In [ ]:
!pip install -q transformers peft accelerate wandb


In [ ]:
import wandb
wandb.login()


## 2. Config — the only cell to edit between experiments

In [ ]:
from training.config import RunConfig

# Every field is captured in runs/<run_id>/config.json and the registry,
# so no two runs are ever ambiguous about what produced them.
config = RunConfig(
    tag="run1-cosent-graded",                      # short label, becomes part of the run_id

    # ── LoRA: literature-backed settings (see EXPERIMENTS.md for sources) ──
    lora_r=16,                                     # rank ~irrelevant at this data scale
    lora_alpha=32,                                 # alpha = 2r (Raschka best practice)
    lora_dropout=0.1,
    target_modules=["Wqkv", "Wo", "Wi"],           # ALL linear layers — attention-only
                                                   # LoRA significantly underperforms
    # use_rslora=True,                             # only worth it if ablating r >= 64
    # target_modules=["Wqkv", "Wo", "Wi", "tok_embeddings"],  # infosec variant
    # modules_to_save=["attn_norm", "mlp_norm", "emb_norm"],  # infosec: unfreeze norms
    # init_from_adapter="/path/to/existing/adapter",  # continue training a saved adapter

    # ── Loss: run sequence — run1 cosent_log_ratio, run2 cosent_plus_head,
    #    run3 dynamic_margin_softplus_capped, then old losses only as baselines ──
    # "cosent_log_ratio"    — graded ranking, order-violations only (START HERE)
    # "cosent_plus_head"    — + downweighted normalized head regression
    # "dynamic_margin_softplus"         — trainer_3's loss, unchanged (baseline)
    # "dynamic_margin_softplus_capped"  — same, with margin cap + normalized head
    # "dynamic_margin_softplus_infosec" — infosec notebook's rank-margin-0.3 variant
    loss_fn="cosent_log_ratio",
    loss_tau=0.05,                                 # cosent temperature (ablate 0.03/0.1 later)
    base_margin=0.2, loss_alpha=0.5, loss_beta=0.4,  # dynamic_margin_* knobs

    # ── Metric space: False = train the encoder space that eval measures ──
    use_metric_proj=False,                         # True reproduces old notebook behavior

    # ── Data: graded splits (group-aware, decontaminated vs old test sets) ──
    train_file=os.path.join(WORK_DIR, "NumerSense/graded_train.jsonl"),
    val_file=os.path.join(WORK_DIR, "NumerSense/graded_val.jsonl"),
    max_length=128,

    epochs=3,                                      # pick checkpoint by ordering_mean, not loss
    batch_size=32,                                 # LoRA prefers modest batches; 64 ok on A100
    lora_lr=2e-4,                                  # LoRA optimum ~10x full-FT LR (2e-5)
    head_lr=1e-3,
    proj_lr=5e-4,
    weight_decay=0.01,
    seed=42,
    lr_schedule="cosine",                          # "none" = constant LR (old behavior)
    warmup_ratio=0.05,
    grad_clip=1.0,

    save_checkpoint_steps=500,
    ordering_eval_each_epoch=True,                 # log ordering-rho per epoch
    resume=False,                                  # True + resume_run_id="..." to continue a run
    wandb_project="modernbert-numeracy-lora",
    notes="run1: cosent + graded data + encoder-space training",
)


## 3. Train

In [ ]:
from training.train import train_one_run

result = train_one_run(config, WORK_DIR)
result


## 4. Evaluate
Runs triplet metrics on the three test splits plus the ordering suites (Spearman rho
between similarity rank and |log ratio| rank — 1.0 = perfect numeric ordering; the
`monotonic_decay` suite is the near-tie probe where past models fail).

To evaluate a *past* run instead, skip the training cell and set
`run_id`/`adapter_path` by hand here.

In [ ]:
from training.evaluate import evaluate_adapter

run_id = result["run_id"]
adapter_path = os.path.join(WORK_DIR, "runs", run_id, "adapter")
# -- or evaluate any past checkpoint:
# run_id = "20260726-0000_baseline-rerun"
# adapter_path = os.path.join(WORK_DIR, "runs", run_id, "adapter")
# adapter_path = None  # base ModernBERT, no adapter

TRIPLET_FILES = {
    "test_generic": os.path.join(WORK_DIR, "NumerSense/test_generic_extracted.jsonl"),
    "test_same":    os.path.join(WORK_DIR, "NumerSense/test_same_extracted.jsonl"),
    "test_sub":     os.path.join(WORK_DIR, "NumerSense/test_sub_extracted.jsonl"),
}

metrics = evaluate_adapter(
    base_model=config.model_name,
    adapter_path=adapter_path,
    triplet_files=TRIPLET_FILES,
)
{k: v for k, v in metrics.items() if "ordering" in k or "spread" in k}


## 5. Record & compare

In [ ]:
from training.experiment import log_result, load_registry

log_result(WORK_DIR, run_id, config, metrics,
           eval_file=";".join(TRIPLET_FILES.values()))

df = load_registry(WORK_DIR)
cols = ["run_id", "loss_fn", "lora_r", "notes",
        "ordering_mean", "ordering@monotonic_decay", "sim_spread_mean",
        "test_same/triplet_accuracy", "test_generic/recall@5"]
df[[c for c in cols if c in df.columns]].sort_values("ordering_mean", ascending=False)


In [ ]:
# Optional: qualitative look at any suite's full ranking for this model
from training.evaluate import load_eval_model, ordering_scores

model, tokenizer = load_eval_model(config.model_name, adapter_path)
_ = ordering_scores(model, tokenizer, verbose=True)


In [ ]:
from google.colab import runtime
runtime.unassign()
